# Public-release note

This curated notebook contains the original research workflow with execution outputs removed. Bloomberg and other licensed source data are not distributed. To run it, supply compatible files under `data/processed/` as described in `data/README.md`. Any results produced locally depend on the user's licensed data and are not included in this repository.


# Dynamic Allocation: GMV and Risk Parity Rebalancing

This notebook applies buy-and-hold, periodic, threshold, and range-based rebalancing rules to rolling Global Minimum Variance and Risk Parity portfolios in a multi-asset USD universe.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use a white background for print-ready charts
plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "axes.titlecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# Locate the repository root when running from either the project or notebooks directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.multi_asset_usd import select_multi_asset_universe
from src.metrics.performance_metrics import performance_summary
from src.optimization.optimization import rolling_optimized_weights
from src.portfolio.multi_asset_rebalancing import run_multi_asset_rebalancing_strategy

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "gmv_risk_parity"
PLOTS_DIR = PROJECT_ROOT / "plots" / "gmv_risk_parity"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
BOND_MATURITY = 10
INCLUDE_CASH = False
ROLLING_WINDOW = 60
MIN_WEIGHT = 0.0
MAX_WEIGHT = 1.0
THRESHOLD = 0.03
SAMPLES = {
    "paper_sample": ("1987-03-31", "2011-12-31"),
    "post_2011": ("2012-01-31", "2026-05-31"),
    "extended": ("1987-03-31", "2026-05-31"),
}

pd.DataFrame([{
    "bond_maturity": BOND_MATURITY,
    "include_cash": INCLUDE_CASH,
    "rolling_window_months": ROLLING_WINDOW,
    "threshold": THRESHOLD,
    "min_weight": MIN_WEIGHT,
    "max_weight": MAX_WEIGHT,
}])

In [ ]:
# Load USD returns and construct the multi-asset investment universe
all_returns = pd.read_csv(PROCESSED_DIR / "multi_asset_usd_returns.csv", parse_dates=["Date"])
universe = select_multi_asset_universe(all_returns, bond_maturity=BOND_MATURITY, include_cash=INCLUDE_CASH)
asset_cols = list(universe.columns.drop("Date"))
rf = all_returns[["Date", "Cash_USD_Return"]]

coverage = pd.DataFrame([{
    "Start": universe["Date"].min(),
    "End": universe["Date"].max(),
    "Rows": len(universe),
    "Assets": len(asset_cols),
}])
coverage

In [ ]:
# Estimate dynamic target weights using only the preceding 60 months
target_weights = {
    "GMV": rolling_optimized_weights(universe, "gmv", ROLLING_WINDOW, min_weight=MIN_WEIGHT, max_weight=MAX_WEIGHT),
    "Risk Parity": rolling_optimized_weights(universe, "risk_parity", ROLLING_WINDOW, min_weight=MIN_WEIGHT, max_weight=MAX_WEIGHT),
}

weights_coverage = pd.DataFrame(
    [
        {"Method": method, "Start": w["Date"].min(), "End": w["Date"].max(), "Rows": len(w)}
        for method, w in target_weights.items()
    ]
)
weights_coverage

In [ ]:
# Transaction costs: 10 bps for equity/alternatives and 5 bps for bonds
transaction_costs = {
    asset: 0.0005 if "Bond" in asset else 0.001
    for asset in asset_cols
}
pd.DataFrame([{"Asset": asset, "Transaction_Cost": cost} for asset, cost in transaction_costs.items()])

In [ ]:
strategies = {
    "Buy-and-hold": {"strategy": "buy_and_hold"},
    "Yearly periodic": {"strategy": "periodic", "frequency": "Y"},
    "Quarterly periodic": {"strategy": "periodic", "frequency": "Q"},
    "Monthly periodic": {"strategy": "periodic", "frequency": "M"},
    "Yearly threshold": {"strategy": "threshold", "frequency": "Y"},
    "Quarterly threshold": {"strategy": "threshold", "frequency": "Q"},
    "Monthly threshold": {"strategy": "threshold", "frequency": "M"},
    "Yearly range": {"strategy": "range", "frequency": "Y"},
    "Quarterly range": {"strategy": "range", "frequency": "Q"},
    "Monthly range": {"strategy": "range", "frequency": "M"},
}

pd.DataFrame([{"Strategy": name, **params} for name, params in strategies.items()])

In [ ]:
def run_strategy(method: str, strategy_name: str, params: dict) -> pd.DataFrame:
    kwargs = {
        "strategy": params["strategy"],
        "returns": universe,
        "target_weights": target_weights[method],
        "threshold": THRESHOLD,
        "transaction_costs": transaction_costs,
    }
    if params["strategy"] != "buy_and_hold":
        kwargs["frequency"] = params["frequency"]

    # Retain method and strategy labels in the combined monthly result table
    return run_multi_asset_rebalancing_strategy(**kwargs).assign(Method=method, Strategy=strategy_name)


results = pd.concat(
    [run_strategy(method, name, params) for method in target_weights for name, params in strategies.items()],
    ignore_index=True,
)
results[["Date", "Method", "Strategy", "Portfolio_Return", "Portfolio_Value", "Turnover", "Transaction_Cost"]].head()

In [ ]:
def summarize_strategy(df: pd.DataFrame) -> dict[str, float]:
    merged = df.merge(rf, on="Date", how="left")
    perf = performance_summary(merged["Portfolio_Return"], rf=merged["Cash_USD_Return"], target=0)
    return {
        "Ann_Return": perf["ann_return"],
        "Ann_Volatility": perf["ann_vol"],
        "Sharpe": perf["sharpe"],
        "Sortino": perf["sortino"],
        "Omega": perf["omega"],
        "Max_Drawdown": perf["max_drawdown"],
        "Turnover": df["Turnover"].sum(),
        "Avg_Turnover": df["Turnover"].mean(),
        "Transaction_Cost": df["Transaction_Cost"].sum(),
        "Rebalanced_Months": int(df["Rebalanced"].sum()),
    }


summary = pd.DataFrame(
    [
        {"Method": method, "Strategy": strategy, **summarize_strategy(g)}
        for (method, strategy), g in results.groupby(["Method", "Strategy"])
    ]
)

strategy_order = list(strategies)
summary["Strategy"] = pd.Categorical(summary["Strategy"], strategy_order, ordered=True)
summary = summary.sort_values(["Method", "Strategy"]).reset_index(drop=True)
summary

In [ ]:
# Compute the same metrics separately by historical sample
sample_summary = pd.DataFrame(
    [
        {
            "Sample": sample,
            "Method": method,
            "Strategy": strategy,
            **summarize_strategy(g[g["Date"].between(start, end)]),
        }
        for sample, (start, end) in SAMPLES.items()
        for (method, strategy), g in results.groupby(["Method", "Strategy"])
    ]
)

sample_summary["Strategy"] = pd.Categorical(sample_summary["Strategy"], strategy_order, ordered=True)
sample_summary = sample_summary.sort_values(["Sample", "Method", "Strategy"]).reset_index(drop=True)
sample_summary.head()

In [ ]:
def panel(metric: str, scale: float = 1) -> pd.DataFrame:
    out = summary.pivot(index="Strategy", columns="Method", values=metric).loc[strategy_order]
    return (out * scale).round(4)


panel_return = panel("Ann_Return", 100)
panel_volatility = panel("Ann_Volatility", 100)
panel_sharpe = panel("Sharpe")
panel_turnover = panel("Turnover")

panel_return

In [ ]:
def sample_panel(metric: str, scale: float = 1) -> pd.DataFrame:
    out = sample_summary.pivot(index=["Sample", "Strategy"], columns="Method", values=metric)
    order = pd.MultiIndex.from_product([SAMPLES.keys(), strategy_order], names=["Sample", "Strategy"])
    return (out.reindex(order) * scale).round(4)


sample_panel_return = sample_panel("Ann_Return", 100)
sample_panel_sharpe = sample_panel("Sharpe")
sample_panel_drawdown = sample_panel("Max_Drawdown", 100)
sample_panel_turnover = sample_panel("Turnover")

sample_panel_sharpe

In [ ]:
panel_sharpe

In [ ]:
panel_turnover

In [ ]:
# Save monthly results and historical summaries
results.to_csv(RESULTS_DIR / "gmv_risk_parity_rebalancing_monthly.csv", index=False)
summary.to_csv(RESULTS_DIR / "gmv_risk_parity_rebalancing_summary.csv", index=False)
sample_summary.to_csv(RESULTS_DIR / "gmv_risk_parity_rebalancing_sample_summary.csv", index=False)

with pd.ExcelWriter(RESULTS_DIR / "gmv_risk_parity_rebalancing.xlsx", engine="openpyxl") as writer:
    coverage.to_excel(writer, sheet_name="coverage", index=False)
    weights_coverage.to_excel(writer, sheet_name="weights_coverage", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    sample_summary.to_excel(writer, sheet_name="sample_summary", index=False)
    panel_return.to_excel(writer, sheet_name="ann_return_pct")
    panel_volatility.to_excel(writer, sheet_name="ann_vol_pct")
    panel_sharpe.to_excel(writer, sheet_name="sharpe")
    panel_turnover.to_excel(writer, sheet_name="turnover")
    sample_panel_return.to_excel(writer, sheet_name="sample_ann_return_pct")
    sample_panel_sharpe.to_excel(writer, sheet_name="sample_sharpe")
    sample_panel_drawdown.to_excel(writer, sheet_name="sample_drawdown_pct")
    sample_panel_turnover.to_excel(writer, sheet_name="sample_turnover")

RESULTS_DIR

In [ ]:
def plot_cumulative(method: str) -> None:
    wealth = results[results["Method"].eq(method)].pivot(index="Date", columns="Strategy", values="Portfolio_Value")
    ax = wealth[strategy_order].plot(figsize=(11, 5), linewidth=1.6)
    ax.set_title(f"Cumulative performance - {method}")
    ax.set_xlabel("Data")
    ax.set_ylabel("Portfolio value")
    ax.grid(alpha=0.25)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"{method.lower().replace(' ', '_')}_rebalancing_cumulative.png", dpi=150, facecolor="white")
    plt.show()


for method in target_weights:
    plot_cumulative(method)

In [ ]:
ax = summary.pivot(index="Strategy", columns="Method", values="Sharpe").loc[strategy_order].plot.bar(figsize=(11, 5))
ax.set_title("Sharpe ratio by strategy")
ax.set_xlabel("Strategy")
ax.set_ylabel("Sharpe ratio")
ax.legend(frameon=False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "gmv_risk_parity_rebalancing_sharpe.png", dpi=150, facecolor="white")
plt.show()

In [ ]:
# Compact comparison of strategy Sharpe ratios across samples
fig, axes = plt.subplots(1, len(SAMPLES), figsize=(16, 5), sharey=True)

for ax, sample in zip(axes, SAMPLES):
    data = sample_summary[sample_summary["Sample"].eq(sample)].pivot(index="Strategy", columns="Method", values="Sharpe").loc[strategy_order]
    data.plot.bar(ax=ax, width=0.8)
    ax.set_title(sample)
    ax.set_xlabel("")
    ax.set_ylabel("Sharpe ratio" if ax is axes[0] else "")
    ax.legend(frameon=False if ax is axes[-1] else True)
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_ha("right")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "gmv_risk_parity_sample_sharpe.png", dpi=150, facecolor="white")
plt.show()